In [1]:
import pandas as pd
import pyecharts.options as opts
from pyecharts.charts import Line
from pyecharts.globals import CurrentConfig, NotebookType
from pyecharts.commons.utils import JsCode  

# 适配Jupyter Lab环境
CurrentConfig.NOTEBOOK_TYPE = NotebookType.JUPYTER_LAB

In [3]:

# ---------------------- 1. 读取月度情绪评分数据 ----------------------
# 修改数据文件路径
monthly_data_path = r"D:\0桌面文件\课程任务\DSAI\期末：所有模型月度情感评分.xlsx"
df = pd.read_excel(monthly_data_path)
df['month_dt'] = pd.to_datetime(df['month'])
df = df.sort_values('month_dt').drop('month_dt', axis=1).reset_index(drop=True)

# ---------------------- 2. 定义可视化列配置（按分值分组） ----------------------
# 组1：所有中心化的 + snownlp（训练前/后）相关（值范围0.5左右）→ 绑定主y轴（index=0）
group1_series = [
    ("hh中心化", "hh中心化"),
    ("snownlp原始得分", "snownlp原始得分"),
    ("snownlp中心化", "snownlp中心化"),
    ("snownlp标题修正", "snownlp标题修正"),
    ("训练snownlp原始得分", "训练snownlp原始得分"),
    ("训练snownlp中心化", "训练snownlp中心化"),
    ("训练snownlp标题修正", "训练snownlp标题修正"),
    ("svm中心化", "svm中心化"),
    ("nb中心化", "nb中心化"),
]

# 组2：其余列（值偏小）→ 绑定副y轴（index=1）
group2_series = [
    ("hh原始得分", "hh原始得分"),
    ("svm原始得分", "svm原始得分"),
    ("svm标题修正", "svm标题修正"),
    ("nb原始得分", "nb原始得分"),
    ("nb标题修正", "nb标题修正"),
]

# 合并校验列是否存在
score_series = group1_series + group2_series
x_data = df['month'].tolist()
missing_cols = [col for col, _ in score_series if col not in df.columns]
if missing_cols:
    raise ValueError(f"数据缺失列：{missing_cols}，请检查月度数据文件")

# ---------------------- 3. 构建折线图 ----------------------
line = (
    Line(init_opts=opts.InitOpts(width="1100px", height="700px", theme="white"))
    .add_xaxis(xaxis_data=x_data)
    # 扩展副y轴（仅1个，用于低值系列）
    .extend_axis(
        yaxis=opts.AxisOpts(
            name="低分值系列",
            position="right",
            offset=0,
            is_scale=True,
            splitline_opts=opts.SplitLineOpts(is_show=True, linestyle_opts=opts.LineStyleOpts(type_="dashed")),
            axislabel_opts=opts.LabelOpts(formatter=JsCode("function(value) {return value.toFixed(4);}"))
        )
    )
)

# 按分组添加系列
# 组1（0.5分值区间）→ 主y轴（index=0）
for col, name in group1_series:
    line.add_yaxis(
        series_name=name,
        y_axis=df[col].tolist(),
        label_opts=opts.LabelOpts(is_show=False),
        is_smooth=True,
        symbol="none",
        yaxis_index=0
    )

# 组2（低分值）→ 副y轴（index=1）
for col, name in group2_series:
    line.add_yaxis(
        series_name=name,
        y_axis=df[col].tolist(),
        label_opts=opts.LabelOpts(is_show=False),
        is_smooth=True,
        symbol="none",
        yaxis_index=1
    )

# ---------------------- 4. 全局配置 ----------------------
line.set_global_opts(
    title_opts=opts.TitleOpts(
        title="所有模型月度情感评分趋势",
        pos_left="center",
        title_textstyle_opts=opts.TextStyleOpts(font_size=16)
    ),
    xaxis_opts=opts.AxisOpts(
        type_="category",
        axislabel_opts=opts.LabelOpts(rotate=45),
        splitline_opts=opts.SplitLineOpts(is_show=False)
    ),
    # 主y轴配置（0.5分值区间系列）
    yaxis_opts=opts.AxisOpts(
        name="0.5分值区间系列",
        is_scale=True,
        splitline_opts=opts.SplitLineOpts(is_show=True, linestyle_opts=opts.LineStyleOpts(type_="dashed")),
        axislabel_opts=opts.LabelOpts(
            formatter=JsCode("function(value) {return value.toFixed(4);}")
        )
    ),
    tooltip_opts=opts.TooltipOpts(
        trigger="axis",
        axis_pointer_type="cross",
    ),
    toolbox_opts=opts.ToolboxOpts(
        pos_left="85%",
        pos_top="5%",
        feature={
            "restore": {},
            "saveAsImage": {},
            "dataView": {},
            "dataZoom": {"yAxisIndex": "none"},
            "brush": opts.ToolBoxFeatureBrushOpts(
                type_=["rect", "lineX", "keep", "clear"]
            )
        }
    ),
    legend_opts=opts.LegendOpts(
        type_="scroll",
        orient='horizontal',
        pos_left='center',
        pos_top='bottom',
        selector=True,
        selector_position="end",
        selector_item_gap=10,
        selector_button_gap=15,
    ),
    brush_opts=opts.BrushOpts(
        brush_link="all",
        x_axis_index=0,
        out_of_brush={"colorAlpha": 0.1}
    )
)

# 渲染图表（Jupyter Lab环境）
line.load_javascript() 


In [5]:
line.render_notebook()

- 5种方法原始评分：整体趋势上有一定的一致性（部分区间表现相同或类似的变化趋势），但也有一定区别，不完全一致。

图像上表现出的一致趋势的主要是：1.整体上呈现下降趋势，反映出近五年社会情绪的衰落；2.22年（年初年中）的时间段情绪评分剧烈下降；3.24年末至25年末，情绪趋于稳定，部分方法表现出一些上升趋势，可能表明社会情绪有所回暖。
- 对比训练前后snownlp的结果：人工评分数据训练后，情绪评分变动性更加明显，可能因为人工评分训练后snownlp能捕获更多情绪词（尤其是B站弹幕特有的词汇）
- 训练snownlp原始、中心化、标题修正：中心化、标题修正后，情绪评分会有所增大或减少，但对整体变化趋势影响不大。
- 21年7月前（尤其19-20年）较平稳，在长期上（整体上）没有表现明显的趋势。但是。。。

In [7]:
import pandas as pd
from pyecharts import options as opts
from pyecharts.charts import Line
from pyecharts.commons.utils import JsCode

# ---------------------- 1. 读取月度情绪评分数据 ----------------------
monthly_data_path = r"D:\0桌面文件\课程任务\DSAI\期末：所有模型月度情感评分+经济指标.xlsx"
df = pd.read_excel(monthly_data_path)
df['month_dt'] = pd.to_datetime(df['month'])
df = df.sort_values('month_dt').drop('month_dt', axis=1).reset_index(drop=True)

# ---------------------- 2. 定义可视化列配置（按分组划分） ----------------------
# 组1：情绪中心化指标（绑定主y轴 index=0）
emotion_series = [
    ("hh中心化", "hh中心化"),
    ("snownlp中心化", "snownlp中心化"),
    ("训练snownlp中心化", "训练snownlp中心化"),
    ("svm中心化", "svm中心化"),
    ("nb中心化", "nb中心化"),
]
# 组2：宏观经济指标（绑定副y轴1 index=1）
economy_series = [
    ("宏观经济景气指数：一致指数", "宏观经济景气指数：一致指数"),
    ("居民消费价格指数", "居民消费价格指数"),
    ("工业生产者出厂价格指数", "工业生产者出厂价格指数"),
]
# 组3：HS300（绑定副y轴2 index=2）
hs300_series = [
    ("HS300", "HS300"),
]
# 组4：社会消费品零售总额（绑定副y轴3 index=3）
retail_series = [
    ("社会消费品零售总额当期值(亿元)", "社会消费品零售总额当期值(亿元)"),
]

# 合并校验（新增retail_series）
score_series = emotion_series + economy_series + hs300_series + retail_series
x_data = df['month'].tolist()
missing_cols = [col for col, _ in score_series if col not in df.columns]
if missing_cols:
    raise ValueError(f"数据缺失列：{missing_cols}，请检查月度数据文件")

# ---------------------- 3. 构建折线图 ----------------------
line = (
    Line(init_opts=opts.InitOpts(width="1100px", height="700px", theme="white"))
    .add_xaxis(xaxis_data=x_data)
    .extend_axis(
        yaxis=opts.AxisOpts(
            name="宏观经济指标",
            position="left",
            offset=50,  # 偏移主y轴避免重叠
            is_scale=True,
            splitline_opts=opts.SplitLineOpts(is_show=True, linestyle_opts=opts.LineStyleOpts(type_="dashed")),
            axislabel_opts=opts.LabelOpts(formatter=JsCode("function(value) {return value.toFixed(4);}"))
        )
    )
    .extend_axis(
        yaxis=opts.AxisOpts(
            name="HS300指数",
            position="right",
            offset=0,
            is_scale=True,
            splitline_opts=opts.SplitLineOpts(is_show=True, linestyle_opts=opts.LineStyleOpts(type_="dashed")),
            axislabel_opts=opts.LabelOpts(formatter=JsCode("function(value) {return value.toFixed(4);}"))
        )
    )
    .extend_axis(  # 新增：社会消费品零售总额y轴（副y轴3）
        yaxis=opts.AxisOpts(
            name="社会消费品零售总额(亿元)",
            position="right",
            offset=50,  # 偏移HS300的y轴避免重叠
            is_scale=True,
            splitline_opts=opts.SplitLineOpts(is_show=True, linestyle_opts=opts.LineStyleOpts(type_="dashed")),
            axislabel_opts=opts.LabelOpts(formatter=JsCode("function(value) {return value.toFixed(4);}"))
        )
    )
)
# 按分组添加系列
# 情绪中心化系列 → 主y轴（index=0）
for col, name in emotion_series:
    line.add_yaxis(
        series_name=name,
        y_axis=df[col].tolist(),
        label_opts=opts.LabelOpts(is_show=False),
        is_smooth=True,
        symbol="none",
        yaxis_index=0
    )
# 宏观经济系列 → 副y轴1（index=1）
for col, name in economy_series:
    line.add_yaxis(
        series_name=name,
        y_axis=df[col].tolist(),
        label_opts=opts.LabelOpts(is_show=False),
        is_smooth=True,
        symbol="none",
        yaxis_index=1
    )
# HS300系列 → 副y轴2（index=2）
for col, name in hs300_series:
    line.add_yaxis(
        series_name=name,
        y_axis=df[col].tolist(),
        label_opts=opts.LabelOpts(is_show=False),
        is_smooth=True,
        symbol="none",
        yaxis_index=2
    )
# 新增：社会消费品零售总额系列 → 副y轴3（index=3）
for col, name in retail_series:
    line.add_yaxis(
        series_name=name,
        y_axis=df[col].tolist(),
        label_opts=opts.LabelOpts(is_show=False),
        is_smooth=True,
        symbol="none",
        yaxis_index=3
    )

# ---------------------- 4. 全局配置（完全沿用首次代码风格） ----------------------
line.set_global_opts(
    title_opts=opts.TitleOpts(
        title="月度情绪中心化指标 vs 宏观经济指标",
        pos_left="center",
        title_textstyle_opts=opts.TextStyleOpts(font_size=16)
    ),
    xaxis_opts=opts.AxisOpts(
        type_="category",
        axislabel_opts=opts.LabelOpts(rotate=45),
        splitline_opts=opts.SplitLineOpts(is_show=False)
    ),
    yaxis_opts=opts.AxisOpts(
        name="情绪中心化指标",
        is_scale=True,
        splitline_opts=opts.SplitLineOpts(is_show=True, linestyle_opts=opts.LineStyleOpts(type_="dashed")),
        axislabel_opts=opts.LabelOpts(
            formatter=JsCode("function(value) {return value.toFixed(4);}")
        )
    ),
    tooltip_opts=opts.TooltipOpts(
        trigger="axis",
        axis_pointer_type="cross",
    ),
    toolbox_opts=opts.ToolboxOpts(
        pos_left="85%",
        pos_top="5%",
        feature={
            "restore": {},
            "saveAsImage": {},
            "dataView": {},
            "dataZoom": {"yAxisIndex": "none"},
            "brush": opts.ToolBoxFeatureBrushOpts(
                type_=["rect", "lineX", "keep", "clear"]
            )
        }
    ),
    legend_opts=opts.LegendOpts(
        type_="scroll",
        orient='horizontal',
        pos_left='center',
        pos_top='bottom',
        selector=True,
        selector_position="end",
        selector_item_gap=10,
        selector_button_gap=15,
    ),
    brush_opts=opts.BrushOpts(
        brush_link="all",
        x_axis_index=0,
        out_of_brush={"colorAlpha": 0.1}
    )
)

line.render_notebook()

- 经济指标交叉验证：除零售总额指标外，其余宏观经济指标长期趋势较为一致，一个显著的特点是：中间20-22年有一段快速上升又快速下降的过程，可能是由疫情叠加出口带动的；另一点是24年至25年趋于平稳（HS300有增长趋势）
- 与情绪指标联系（训练后snownlp,svm）：情绪指标在22年至23年初一个快速下降的过程，之后又趋于稳定，与经济指标的趋势一致。
- 但是，19年-21年中，情绪指标与经济指标的在长期趋势上的一致性不是很好，经济指标快速上升的过程并没有在情绪指标中反映出来。为什么？
- 只保留21年7月后的，整体趋势的一致性很好。
- 零售总额：虽然整体趋势上不明显，但短期波动似乎有一定相关性。